# Ratings-Based Movie Recommendation System

**Original project:** 2022  
**Portfolio version:** curated for GitHub

This notebook implements an item-based collaborative filtering recommender.
User ratings are represented as a sparse movie-user matrix and movie similarity
is measured using cosine distance with k-nearest neighbors.

A row-index mapping issue in the original 2022 notebook has been corrected in
this portfolio version so that movie metadata is aligned with the corresponding
rows in the ratings matrix.

## 1. Imports

In [ ]:
import difflib
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

## 2. Load Movie and Rating Data

In [ ]:
movies = pd.read_csv(
    "../data/movies.csv",
    usecols=["movieId", "title"],
    dtype={"movieId": "int32", "title": "str"}
)

ratings = pd.read_csv(
    "../data/ratings.csv",
    usecols=["userId", "movieId", "rating"],
    dtype={"userId": "int32", "movieId": "int32", "rating": "float32"}
)

print("Movies:", len(movies))
print("Ratings:", len(ratings))
print("Users:", ratings["userId"].nunique())

## 3. Build the Movie-User Matrix

In [ ]:
movie_user_matrix = ratings.pivot_table(
    index="movieId",
    columns="userId",
    values="rating",
    fill_value=0
)

sparse_movie_user_matrix = csr_matrix(movie_user_matrix.values)

print("Matrix shape:", movie_user_matrix.shape)

## 4. Train the Nearest-Neighbors Model

In [ ]:
knn_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute"
)

knn_model.fit(sparse_movie_user_matrix)

## 5. Recommendation Function

In [ ]:
def recommend_by_ratings(movie_name, n_recommendations=10):
    titles = movies["title"].tolist()
    matches = difflib.get_close_matches(movie_name, titles, n=1)

    if not matches:
        return pd.DataFrame(columns=["title", "cosine_similarity"])

    selected_title = matches[0]
    movie_id = movies.loc[
        movies["title"] == selected_title, "movieId"
    ].iloc[0]

    if movie_id not in movie_user_matrix.index:
        return pd.DataFrame(columns=["title", "cosine_similarity"])

    matrix_row = movie_user_matrix.index.get_loc(movie_id)

    distances, indices = knn_model.kneighbors(
        sparse_movie_user_matrix[matrix_row],
        n_neighbors=n_recommendations + 1
    )

    recommendation_ids = movie_user_matrix.index[indices.flatten()]
    recommendations = []

    for rec_id, distance in zip(recommendation_ids, distances.flatten()):
        if rec_id == movie_id:
            continue

        title_match = movies.loc[movies["movieId"] == rec_id, "title"]
        if title_match.empty:
            continue

        recommendations.append(
            (title_match.iloc[0], 1 - distance)
        )

        if len(recommendations) == n_recommendations:
            break

    return pd.DataFrame(
        recommendations,
        columns=["title", "cosine_similarity"]
    )

## 6. Example: *Iron Man (2008)*

In [ ]:
recommend_by_ratings("Iron Man (2008)", n_recommendations=10)

### Interpretation

Unlike the content-based model, this method recommends movies from patterns in
user ratings rather than movie metadata. Similarity therefore reflects overlap
in audience preferences.

With the corrected movie-to-matrix mapping, *Iron Man (2008)* is associated with
titles such as *The Dark Knight (2008)*, *WALL·E (2008)*, *The Avengers (2012)*,
*Iron Man 2 (2010)*, *Avatar (2009)*, and *Batman Begins (2005)*.

## Note on the Original 2022 Notebook

The original notebook used the row number returned from the movie metadata table
directly as the row number of the pivoted ratings matrix. Those two row orders
are not guaranteed to match. This portfolio version explicitly maps the selected
movie to its `movieId` and then locates the corresponding matrix row before
querying nearest neighbors.

This correction improves the implementation without changing the underlying
collaborative-filtering method used in the original project.